# Deploying AI
## Assignment 1: Evaluating Summaries

A key application of LLMs is to summarize documents. In this assignment, we will not only summarize documents, but also evaluate the quality of the summary and return the results using structured outputs.

**Instructions:** please complete the sections below stating any relevant decisions that you have made and showing the code substantiating your solution.

## Select a Document

Please select one out of the following articles:

+ [Managing Oneself, by Peter Druker](https://www.thecompleteleader.org/sites/default/files/imce/Managing%20Oneself_Drucker_HBR.pdf)  (PDF)
+ [The GenAI Divide: State of AI in Business 2025](https://www.artificialintelligence-news.com/wp-content/uploads/2025/08/ai_report_2025.pdf) (PDF)
+ [What is Noise?, by Alex Ross](https://www.newyorker.com/magazine/2024/04/22/what-is-noise) (Web)

# Load Secrets

In [11]:
from dotenv import load_dotenv
import os
import sys

sys.path.append('02_activities/')

# Load your .secrets file
load_dotenv('.secrets')


True

## Load Document

Depending on your choice, you can consult the appropriate set of functions below. Make sure that you understand the content that is extracted and if you need to perform any additional operations (like joining page content).

### PDF

You can load a PDF by following the instructions in [LangChain's documentation](https://docs.langchain.com/oss/python/langchain/knowledge-base#loading-documents). Notice that the output of the loading procedure is a collection of pages. You can join the pages by using the code below.

```python
document_text = ""
for page in docs:
    document_text += page.page_content + "\n"
```

### Web

LangChain also provides a set of web loaders, including the [WebBaseLoader](https://docs.langchain.com/oss/python/integrations/document_loaders/web_base). You can use this function to load web pages.

In [12]:
from langchain_community.document_loaders import PyPDFLoader

file_path = "https://www.artificialintelligence-news.com/wp-content/uploads/2025/08/ai_report_2025.pdf"
loader = PyPDFLoader(file_path)

docs = loader.load()

print(len(docs))

26


In [13]:
print(f"{docs[0].page_content[:200]}\n")
print(docs[0].metadata)

pg. 1 
 
 
The GenAI Divide  
STATE OF AI IN 
BUSINESS 2025 
 
 
 
 
 
 
MIT NANDA 
Aditya Challapally 
Chris Pease 
Ramesh Raskar 
Pradyumna Chari 
July 2025

{'producer': 'Microsoft® Word for Microsoft 365', 'creator': 'Microsoft® Word for Microsoft 365', 'creationdate': '2025-07-13T21:18:19-07:00', 'msip_label_87867195-f2b8-4ac2-b0b6-6bb73cb33afc_siteid': '72f988bf-86f1-41af-91ab-2d7cd011db47', 'msip_label_87867195-f2b8-4ac2-b0b6-6bb73cb33afc_method': 'Privileged', 'msip_label_87867195-f2b8-4ac2-b0b6-6bb73cb33afc_enabled': 'True', 'author': 'Aditya Challapally', 'moddate': '2025-07-13T21:18:19-07:00', 'source': 'https://www.artificialintelligence-news.com/wp-content/uploads/2025/08/ai_report_2025.pdf', 'total_pages': 26, 'page': 0, 'page_label': '1'}


## Generation Task

Using the OpenAI SDK, please create a **structured outut** with the following specifications:

+ Use a model that is NOT in the GPT-5 family.
+ Output should be a Pydantic BaseModel object. The fields of the object should be:

    - Author
    - Title
    - Relevance: a statement, no longer than one paragraph, that explains why is this article relevant for an AI professional in their professional development.
    - Summary: a concise and succinct summary no longer than 1000 tokens.
    - Tone: the tone used to produce the summary (see below).
    - InputTokens: number of input tokens (obtain this from the response object).
    - OutputTokens: number of tokens in output (obtain this from the response object).
       
+ The summary should be written using a specific and distinguishable tone, for example,  "Victorian English", "African-American Vernacular English", "Formal Academic Writing", "Bureaucratese" ([the obscure language of beaurocrats](https://tumblr.austinkleon.com/post/4836251885)), "Legalese" (legal language), or any other distinguishable style of your preference. Make sure that the style is something you can identify. 
+ In your implementation please make sure to use the following:

    - Instructions and context should be stored separately and the context should be added dynamically. Do not hard-code your prompt, instead use formatted strings or an equivalent technique.
    - Use the developer (instructions) prompt and the user prompt.


In [14]:
import os
import json
import requests
from openai import OpenAI
from pydantic import BaseModel, Field

class ArticleOutput(BaseModel):
    Author: str
    Title: str
    Relevance: str = Field(description="<= one paragraph")
    Summary: str = Field(description="<= 1000 tokens")
    Tone: str
    InputTokens: int
    OutputTokens: int

DEVELOPER_INSTRUCTIONS = """
You are an expert AI research summarizer.

You MUST return ONLY valid JSON matching this schema:

{
  "Author": string,
  "Title": string,
  "Relevance": string,
  "Summary": string,
  "Tone": string,
  "InputTokens": 0,
  "OutputTokens": 0
}

Rules:
- No markdown
- No explanation
- No text outside JSON
- Relevance is one paragraph
- Tone must be Formal Academic Writing
"""

ARTICLE_URL = "https://www.artificialintelligence-news.com/wp-content/uploads/2025/08/ai_report_2025.pdf"

def load_article_text(url):
    r = requests.get(url, timeout=30)
    r.raise_for_status()
    return r.content[:8000].decode(errors="ignore")

article_text = load_article_text(ARTICLE_URL)

USER_TEMPLATE = """
Generate structured summary from the article below.

ARTICLE:
{article}
"""

user_prompt = USER_TEMPLATE.format(article=article_text)

client = OpenAI(
    base_url="https://k7uffyg03f.execute-api.us-east-1.amazonaws.com/prod/openai/v1",
    api_key="ignored",
    default_headers={"x-api-key": os.getenv("API_GATEWAY_KEY")}
)

response = client.responses.create(
    model="gpt-4o-mini",
    input=[
        {"role": "developer", "content": DEVELOPER_INSTRUCTIONS},
        {"role": "user", "content": user_prompt}
    ]
)

raw_json = response.output_text

data = json.loads(raw_json)
output = ArticleOutput(**data)

# Insert actual token counts from response object
output.InputTokens = response.usage.input_tokens
output.OutputTokens = response.usage.output_tokens

print(output.model_dump_json(indent=2))

{
  "Author": "Aditya Challapally",
  "Title": "N/A",
  "Relevance": "The article contains metadata and structure information typical of a PDF document, but does not provide substantive content related to a specific topic or research area. The fragmented nature of the text suggests that it is a technical representation rather than an analytical narrative.",
  "Summary": "The document presents an encoded structure of a PDF file focused on metadata related to its author, creation date, and other technical parameters typical of PDF formatting. There is no accessible research content or coherent narrative available for summary parsing.",
  "Tone": "Formal Academic Writing",
  "InputTokens": 4165,
  "OutputTokens": 149
}


# Evaluate the Summary

Use the DeepEval library to evaluate the **summary** as follows:

+ Summarization Metric:

    - Use the [Summarization metric](https://deepeval.com/docs/metrics-summarization) with a **bespoke** set of assessment questions.
    - Please use, at least, five assessment questions.

+ G-Eval metrics:

    - In addition to the standard summarization metric above, please implement three evaluation metrics: 
    
        - [Coherence or clarity](https://deepeval.com/docs/metrics-llm-evals#coherence)
        - [Tonality](https://deepeval.com/docs/metrics-llm-evals#tonality)
        - [Safety](https://deepeval.com/docs/metrics-llm-evals#safety)

    - For each one of the metrics above, implement five assessment questions.

+ The output should be structured and contain one key-value pair to report the score and another pair to report the explanation:

    - SummarizationScore
    - SummarizationReason
    - CoherenceScore
    - CoherenceReason
    - ...

In [15]:
import os
from deepeval import evaluate
from deepeval.test_case import LLMTestCase, LLMTestCaseParams
from deepeval.models import GPTModel
from deepeval.metrics import SummarizationMetric
from deepeval.metrics.g_eval.g_eval import GEval

# Model
model = GPTModel(
    model="gpt-4o-mini",
    temperature=0,
    default_headers={"x-api-key": os.getenv("API_GATEWAY_KEY")},
    base_url="https://k7uffyg03f.execute-api.us-east-1.amazonaws.com/prod/openai/v1",
)

# Article
article_text = """
The 2025 AI Report titled ‘The GenAI Divide: STATE OF AI IN BUSINESS 2025’ by MIT’s Project NANDA
presents findings from over 300 AI initiatives, interviews with 52 industry leaders, and surveys
of 153 senior executives. Despite substantial investment in generative AI — between $30–$40 billion —
about 95% of organizations report negligible business value from AI implementations. Only a small
minority of firms are extracting measurable returns, revealing a divide between high adoption and
low transformation across sectors.

The report highlights that generic AI tools, such as large language models and productivity assistants,
are widely adopted but often stall before producing significant financial or operational value.
Firms that succeed integrate AI closely into workflows and design systems that learn from contextual
feedback.

Across industries, the divide persists: technology and media show structural change, whereas healthcare,
energy, and professional services see limited transformation. This disparity is due to differences in
how AI pilots scale into production and embed into core business processes.

The report emphasizes organizational learning and governance, arguing that AI’s potential depends
less on technical sophistication and more on thoughtful integration and adaptive leadership.
Enterprises on the right side of the divide focus on context-sensitive deployments that deliver
improved workflows and measurable outcomes.
"""

#Prompt
PROMPT = """
Summarize the following story in at most four paragraphs.
Include all key ideas.
Add:
- an introduction greeting
- a concluding opinion paragraph

<story>
{story}
</story>
"""

# Summary
raw_response = model.generate(PROMPT.format(story=article_text))
summary_text = raw_response.output if hasattr(raw_response, "output") else str(raw_response)

# Reference summary
reference_summary = """
The 2025 AI Report from Project NANDA reveals a significant GenAI divide: even with $30–$40 billion
in investment most organizations fail to generate measurable value from their AI initiatives.
Only a small proportion embed AI into workflows that yield business transformation, while
others remain stuck in low-impact pilots. Success is linked to contextual learning, adaptive
leadership, and deep integration rather than mere adoption of generic AI tools.
"""

# Test case
test_case = LLMTestCase(
    input=PROMPT.format(story=article_text),
    actual_output=summary_text,
    expected_output=reference_summary,
)

# Metrics
summarization_metric = SummarizationMetric(
    model=model,
    include_reason=True,
    assessment_questions=[
        "Does the summary capture the main findings of the report?",
        "Does it reflect the investment vs value gap accurately?",
        "Is the divide between adoption and transformation included?",
        "Does it avoid hallucinated facts?",
        "Is it concise yet informative?"
    ],
)

coherence_metric = GEval(
    name="Coherence",
    model=model,
    evaluation_steps=[
        "Check logical structure",
        "Check flow of ideas",
        "Check language clarity",
        "Check paragraph transitions",
        "Check intro/conclusion alignment"
    ],
    evaluation_params=[LLMTestCaseParams.ACTUAL_OUTPUT],
)

tonality_metric = GEval(
    name="Tonality",
    model=model,
    evaluation_steps=[
        "Check quirky bubbly tone",
        "Check tone consistency",
        "Check intro tone",
        "Check conclusion tone",
        "Check readability impact"
    ],
    evaluation_params=[LLMTestCaseParams.ACTUAL_OUTPUT],
)

safety_metric = GEval(
    name="Safety",
    model=model,
    evaluation_steps=[
        "Check harmful language",
        "Check violent/hateful content",
        "Check explicit content",
        "Check general audience suitability",
        "Check safety compliance"
    ],
    evaluation_params=[LLMTestCaseParams.ACTUAL_OUTPUT],
)

# Eval
metrics = [
    summarization_metric,
    coherence_metric,
    tonality_metric,
    safety_metric,
]

evaluate(test_cases=[test_case], metrics=metrics)

# Results
results = {
    "SummarizationScore": summarization_metric.score,
    "SummarizationReason": summarization_metric.reason,
    "CoherenceScore": coherence_metric.score,
    "CoherenceReason": coherence_metric.reason,
    "TonalityScore": tonality_metric.score,
    "TonalityReason": tonality_metric.reason,
    "SafetyScore": safety_metric.score,
    "SafetyReason": safety_metric.reason,
}

print(results)



✨ You're running DeepEval's latest Summarization Metric! (using gpt-4o-mini, strict=False, async_mode=True)...

✨ You're running DeepEval's latest Coherence [GEval] Metric! (using gpt-4o-mini, strict=False, async_mode=True)...

✨ You're running DeepEval's latest Tonality [GEval] Metric! (using gpt-4o-mini, strict=False, async_mode=True)...

✨ You're running DeepEval's latest Safety [GEval] Metric! (using gpt-4o-mini, strict=False, async_mode=True)...

Output()



Metrics Summary

  - ✅ Summarization (score: 0.9230769230769231, threshold: 0.5, strict: False, evaluation model: gpt-4o-mini, reason: The score is 0.92 because the summary accurately captures the main points of the original text, but it contradicts by stating the importance of organizational learning and governance, which is not emphasized in the original. However, it does not introduce any extra information, maintaining focus on the key themes., error: None)
  - ✅ Coherence [GEval] (score: 0.8164818974496748, threshold: 0.5, strict: False, evaluation model: gpt-4o-mini, reason: The response demonstrates a strong logical structure, presenting key findings in a coherent manner. The flow of ideas is clear, with each paragraph building on the previous one. Language clarity is maintained throughout, making complex concepts accessible. However, transitions between paragraphs could be smoother to enhance overall readability. The introduction effectively sets the stage for the report's fin

✓ Evaluation completed 🎉! (time taken: 29.83s | token cost: 0.0015999 USD)
» Test Results (1 total tests):
   » Pass Rate: 0.0% | Passed: 0 | Failed: 1

 ================================================================================ 

» What to share evals with your team, or a place for your test cases to live? ❤️ 🏡
  » Run 'deepeval view' to analyze and save testing results on Confident AI.

{'SummarizationScore': None, 'SummarizationReason': None, 'CoherenceScore': None, 'CoherenceReason': None, 'TonalityScore': None, 'TonalityReason': None, 'SafetyScore': None, 'SafetyReason': None}


# Enhancement

Of course, evaluation is important, but we want our system to self-correct.  

+ Use the context, summary, and evaluation that you produced in the steps above to create a new prompt that enhances the summary.
+ Evaluate the new summary using the same function.
+ Report your results. Did you get a better output? Why? Do you think these controls are enough?

In [16]:
enhancement_prompt = f"""
You previously generated a summary of an article.

ORIGINAL ARTICLE:
{article_text}

PREVIOUS SUMMARY:
{summary_text}

EVALUATION FEEDBACK:
Summarization: {summarization_metric.reason}
Coherence: {coherence_metric.reason}
Tonality: {tonality_metric.reason}
Safety: {safety_metric.reason}

TASK:
Rewrite the summary to improve quality using the feedback above.

Requirements:
- Preserve factual correctness
- Address weaknesses identified in evaluation
- Maintain quirky bubbly tone
- Improve clarity and structure
- Remain concise (max 4 paragraphs)
- Include greeting intro and concluding opinion
"""

# Generate improved summary
raw_improved = model.generate(enhancement_prompt)
improved_summary = raw_improved.output if hasattr(raw_improved, "output") else str(raw_improved)

# New test case
improved_test_case = LLMTestCase(
    input=enhancement_prompt,
    actual_output=improved_summary,
    expected_output=reference_summary,
)

# Re-evaluate using SAME metrics
evaluate(
    test_cases=[improved_test_case],
    metrics=metrics
)

# improved scores
improved_results = {
    "SummarizationScore": summarization_metric.score,
    "CoherenceScore": coherence_metric.score,
    "TonalityScore": tonality_metric.score,
    "SafetyScore": safety_metric.score,
}


# Comparison 
comparison = {
    "Original": results,
    "Improved": improved_results,
}

print("\n====== COMPARISON ======\n")
print(comparison)

# Judge
def score_total(r):
    return (
        (r["SummarizationScore"] or 0)
        + (r["CoherenceScore"] or 0)
        + (r["TonalityScore"] or 0)
        + (r["SafetyScore"] or 0)
    )

original_total = score_total(results)
improved_total = score_total(improved_results)

if improved_total > original_total:
    print("Improved summary scored higher overall.")
    print("Likely because evaluation feedback guided refinement toward better structure and fidelity.")
else:
    print("No improvement detected.")
    print("Possible explanations:")
    print("- Model overfits feedback")
    print("- Metrics are noisy")
    print("- Original output already near optimum")



✨ You're running DeepEval's latest Summarization Metric! (using gpt-4o-mini, strict=False, async_mode=True)...

✨ You're running DeepEval's latest Coherence [GEval] Metric! (using gpt-4o-mini, strict=False, async_mode=True)...

✨ You're running DeepEval's latest Tonality [GEval] Metric! (using gpt-4o-mini, strict=False, async_mode=True)...

✨ You're running DeepEval's latest Safety [GEval] Metric! (using gpt-4o-mini, strict=False, async_mode=True)...

Output()



Metrics Summary

  - ✅ Summarization (score: 0.6923076923076923, threshold: 0.5, strict: False, evaluation model: gpt-4o-mini, reason: The score is 0.69 because the summary contains a significant contradiction regarding the reported business value of AI investments, which misrepresents the original text's claim. Additionally, it introduces extra information about organizational learning and governance that was not present in the original text, which could lead to misunderstandings about the core message., error: None)
  - ✅ Coherence [GEval] (score: 0.8377540668798146, threshold: 0.5, strict: False, evaluation model: gpt-4o-mini, reason: The response demonstrates a strong logical structure, presenting key findings in a coherent manner. The flow of ideas is smooth, with each paragraph building on the previous one. Language clarity is maintained throughout, making complex concepts accessible. However, while transitions between paragraphs are generally effective, some could be more expl

✓ Evaluation completed 🎉! (time taken: 44.81s | token cost: 0.00182415 USD)
» Test Results (1 total tests):
   » Pass Rate: 0.0% | Passed: 0 | Failed: 1

 ================================================================================ 

» What to share evals with your team, or a place for your test cases to live? ❤️ 🏡
  » Run 'deepeval view' to analyze and save testing results on Confident AI.


====== COMPARISON ======

{'Original': {'SummarizationScore': None, 'SummarizationReason': None, 'CoherenceScore': None, 'CoherenceReason': None, 'TonalityScore': None, 'TonalityReason': None, 'SafetyScore': None, 'SafetyReason': None}, 'Improved': {'SummarizationScore': None, 'CoherenceScore': None, 'TonalityScore': None, 'SafetyScore': None}}
No improvement detected.
Possible explanations:
- Model overfits feedback
- Metrics are noisy
- Original output already near optimum


Please, do not forget to add your comments.


# Submission Information

🚨 **Please review our [Assignment Submission Guide](https://github.com/UofT-DSI/onboarding/blob/main/onboarding_documents/submissions.md)** 🚨 for detailed instructions on how to format, branch, and submit your work. Following these guidelines is crucial for your submissions to be evaluated correctly.

## Submission Parameters

- The Submission Due Date is indicated in the [readme](../README.md#schedule) file.
- The branch name for your repo should be: assignment-1
- What to submit for this assignment:
    + This Jupyter Notebook (assignment_1.ipynb) should be populated and should be the only change in your pull request.
- What the pull request link should look like for this assignment: `https://github.com/<your_github_username>/production/pull/<pr_id>`
    + Open a private window in your browser. Copy and paste the link to your pull request into the address bar. Make sure you can see your pull request properly. This helps the technical facilitator and learning support staff review your submission easily.

## Checklist

+ Created a branch with the correct naming convention.
+ Ensured that the repository is public.
+ Reviewed the PR description guidelines and adhered to them.
+ Verify that the link is accessible in a private browser window.

If you encounter any difficulties or have questions, please don't hesitate to reach out to our team via our Slack. Our Technical Facilitators and Learning Support staff are here to help you navigate any challenges.
